# Official Infiltration All-Days PCAP for NIPS System Test

1. memakai filtered PCAP Wednesday yang sudah pernah dibuat jika valid;
2. memakai filtered PCAP Thursday yang sudah ada jika valid;
3. jika filtered PCAP harian belum ada/tidak valid, notebook mencoba membangun ulang dari ZIP raw resmi;
4. menggabungkan PCAP harian menjadi satu file final;
5. memvalidasi bahwa PCAP final memuat dua victim private IP:
   - `172.31.69.24` untuk Wednesday;
   - `172.31.69.13` untuk Thursday.

Notebook ini tidak berhenti dengan traceback kecuali `STRICT_MODE=True`. Jika ada file yang belum tersedia, notebook akan memberi status `NOT_READY` dan menunjukkan file mana yang perlu dicek.

In [1]:
# CELL 1 - Setup
from pathlib import Path
import os, json, re, shutil, subprocess, hashlib, zipfile, datetime, urllib.parse
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

BASE_DIR = Path('/content/drive/MyDrive/CICIDS2018')
PROJECT_DIR = BASE_DIR / 'official_infiltration_alldays_pcap_system_test_ready'
ZIP_DIR = PROJECT_DIR / 'zips'
MEMBER_DIR = PROJECT_DIR / 'selected_members_noerror_final'
OUT_DIR = PROJECT_DIR / 'outputs'
LOG_DIR = PROJECT_DIR / 'logs'
for d in [ZIP_DIR, MEMBER_DIR, OUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# False = notebook tidak berhenti dengan traceback; status akhir akan menjelaskan masalah.
# True = notebook akan raise error kalau final PCAP belum valid.
STRICT_MODE = False

print('BASE_DIR    :', BASE_DIR)
print('PROJECT_DIR :', PROJECT_DIR)
print('ZIP_DIR     :', ZIP_DIR)
print('MEMBER_DIR  :', MEMBER_DIR)
print('OUT_DIR     :', OUT_DIR)
print('STRICT_MODE :', STRICT_MODE)

Mounted at /content/drive
BASE_DIR    : /content/drive/MyDrive/CICIDS2018
PROJECT_DIR : /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready
ZIP_DIR     : /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/zips
MEMBER_DIR  : /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/selected_members_noerror_final
OUT_DIR     : /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs
STRICT_MODE : False


In [2]:
# CELL 2 - Metadata resmi dan path eksplisit
AWS_BUCKET_HOST = 'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com'
AWS_PREFIX_ROOT = 'Original Network Traffic and Log data'
EXPECTED_MIN_ZIP_BYTES = 20_000_000_000

DAYS = [
    {
        'day': 'Wednesday-28-02-2018',
        'attacker_ip': '13.58.225.34',
        'victims': ['18.221.148.137', '172.31.69.24'],
        'victim_private': '172.31.69.24',
        'zip_path': BASE_DIR / 'official_zero_day_infiltration_pcap_claim_ready' / 'zips' / 'Wednesday-28-02-2018_pcap.zip',
        'filtered_candidates': [
            BASE_DIR / 'official_zero_day_infiltration_pcap_claim_ready' / 'outputs' / 'official_infiltration_attacker_victim_filtered.pcap',
            BASE_DIR / 'official_aws_infiltration_raw_pcap' / 'outputs' / 'official_infiltration_attacker_victim_filtered.pcap',
            OUT_DIR / 'filtered_Wednesday-28-02-2018_official_infiltration_attacker_victim_filtered.pcap',
        ],
        'official_windows_text': ['10:50-12:05', '13:42-14:40'],
    },
    {
        'day': 'Thursday-01-03-2018',
        'attacker_ip': '13.58.225.34',
        'victims': ['18.216.254.154', '172.31.69.13'],
        'victim_private': '172.31.69.13',
        'zip_path': BASE_DIR / 'official_infiltration_alldays_pcap_system_test_ready' / 'zips' / 'Thursday-01-03-2018_pcap.zip',
        'filtered_candidates': [
            OUT_DIR / 'official_infiltration_alldays_attacker_victim_filtered.pcap',  # output lama ini terbukti berisi Thursday saja; boleh dipakai sebagai part Thursday jika valid
        ],
        'official_windows_text': ['09:57-10:55', '14:00-15:37'],
    },
]

# Tambahkan semua filtered Thursday hasil cell lama jika ada.
DAYS[1]['filtered_candidates'].extend(sorted(OUT_DIR.glob('filtered_Thursday-01-03-2018*.pcap')))

COMBINED_FINAL_PCAP = OUT_DIR / 'official_infiltration_alldays_attacker_victim_filtered_FINAL.pcap'
COMBINED_COMPAT_PCAP = OUT_DIR / 'official_infiltration_alldays_attacker_victim_filtered.pcap'
MANIFEST_JSON = OUT_DIR / 'official_infiltration_alldays_pcap_system_test_NOERROR_FINAL_manifest.json'

for item in DAYS:
    print('\nDAY:', item['day'])
    print('ZIP:', item['zip_path'], 'exists=', item['zip_path'].exists())
    print('Filtered candidates:')
    for c in item['filtered_candidates']:
        print(' -', c, 'exists=', Path(c).exists())


DAY: Wednesday-28-02-2018
ZIP: /content/drive/MyDrive/CICIDS2018/official_zero_day_infiltration_pcap_claim_ready/zips/Wednesday-28-02-2018_pcap.zip exists= True
Filtered candidates:
 - /content/drive/MyDrive/CICIDS2018/official_zero_day_infiltration_pcap_claim_ready/outputs/official_infiltration_attacker_victim_filtered.pcap exists= True
 - /content/drive/MyDrive/CICIDS2018/official_aws_infiltration_raw_pcap/outputs/official_infiltration_attacker_victim_filtered.pcap exists= True
 - /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/filtered_Wednesday-28-02-2018_official_infiltration_attacker_victim_filtered.pcap exists= False

DAY: Thursday-01-03-2018
ZIP: /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/zips/Thursday-01-03-2018_pcap.zip exists= True
Filtered candidates:
 - /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/official_infiltration_alldays_attack

In [3]:
# CELL 3 - Helpers

def which(cmd):
    return shutil.which(cmd) is not None


def run_cmd(cmd, check=False, capture=True):
    print('\nCMD:', cmd)
    res = subprocess.run(cmd, shell=True, text=True, capture_output=capture)
    if capture:
        if res.stdout:
            print('STDOUT preview:\n', res.stdout[:4000])
        if res.stderr:
            print('STDERR preview:\n', res.stderr[:4000])
    if check and res.returncode != 0:
        raise RuntimeError(f'Command failed rc={res.returncode}: {cmd}')
    return res


def sha256_file(path, chunk_size=1024*1024*8):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def file_info(path, compute_sha256=False):
    path = Path(path)
    out = {'path': str(path), 'exists': path.exists()}
    if path.exists():
        st = path.stat()
        out.update({
            'size_bytes': int(st.st_size),
            'size_kb': round(st.st_size / 1024, 3),
            'size_mb': round(st.st_size / (1024**2), 3),
            'modified_time': datetime.datetime.fromtimestamp(st.st_mtime).isoformat(),
        })
        if compute_sha256:
            out['sha256'] = sha256_file(path)
    return out


def s3_pcap_zip_url(day):
    key = f'{AWS_PREFIX_ROOT}/{day}/pcap.zip'
    return f"{AWS_BUCKET_HOST}/{urllib.parse.quote(key, safe='/')}"


def status_line(ok, msg):
    print(('OK   ' if ok else 'WARN ') + msg)

print('wget:', which('wget'))
print('tshark:', which('tshark'))
print('mergecap:', which('mergecap'))
print('capinfos:', which('capinfos'))
print('tcpdump:', which('tcpdump'))

wget: True
tshark: False
mergecap: False
capinfos: False
tcpdump: False


In [4]:
# CELL 4 - Install tools jika perlu
INSTALL_TOOLS = True
if INSTALL_TOOLS:
    missing = [cmd for cmd in ['tshark', 'mergecap', 'capinfos', 'tcpdump', 'unzip'] if not which(cmd)]
    if missing:
        print('Installing tools:', missing)
        run_cmd('apt-get update -qq', check=False, capture=True)
        run_cmd('DEBIAN_FRONTEND=noninteractive apt-get install -y -qq tshark tcpdump unzip', check=False, capture=True)
    else:
        print('Tools sudah tersedia.')

print('tshark:', shutil.which('tshark'))
print('mergecap:', shutil.which('mergecap'))
print('capinfos:', shutil.which('capinfos'))

Installing tools: ['tshark', 'mergecap', 'capinfos', 'tcpdump']

CMD: apt-get update -qq
STDERR preview:
 W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


CMD: DEBIAN_FRONTEND=noninteractive apt-get install -y -qq tshark tcpdump unzip
STDOUT preview:
 Preconfiguring packages ...
Selecting previously unselected package libpcap0.8:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Rea

In [5]:
# CELL 5 - Fungsi inspeksi PCAP menggunakan tshark

def inspect_pcap_ips(pcap_path):
    pcap_path = Path(pcap_path)
    result = {
        'path': str(pcap_path),
        'exists': pcap_path.exists(),
        'ok': False,
        'packet_rows': 0,
        'observed_ips': [],
        'first_epoch': None,
        'last_epoch': None,
        'error': None,
    }
    if not pcap_path.exists() or pcap_path.stat().st_size <= 0:
        result['error'] = 'file_missing_or_empty'
        return result
    if not which('tshark'):
        result['error'] = 'tshark_missing'
        return result
    cmd = (
        f'tshark -r "{pcap_path}" -T fields '
        '-e frame.time_epoch -e ip.src -e ip.dst '
        '-E header=y -E separator=, -E quote=d'
    )
    res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if res.returncode != 0:
        result['error'] = res.stderr[:1000]
        return result
    from io import StringIO
    try:
        df = pd.read_csv(StringIO(res.stdout))
        result['packet_rows'] = int(len(df))
        if len(df) > 0:
            ips = set(df['ip.src'].dropna().astype(str)).union(set(df['ip.dst'].dropna().astype(str)))
            result['observed_ips'] = sorted(ips)
            result['first_epoch'] = float(pd.to_numeric(df['frame.time_epoch'], errors='coerce').min())
            result['last_epoch'] = float(pd.to_numeric(df['frame.time_epoch'], errors='coerce').max())
            result['ok'] = True
    except Exception as e:
        result['error'] = repr(e)
    return result


def pcap_contains_expected(pcap_path, attacker_ip, victim_private):
    info = inspect_pcap_ips(pcap_path)
    ips = set(info.get('observed_ips', []))
    info['contains_attacker'] = attacker_ip in ips
    info['contains_victim_private'] = victim_private in ips
    info['valid_for_day'] = bool(info['ok'] and info['contains_attacker'] and info['contains_victim_private'])
    return info

print('Inspection helper ready.')

Inspection helper ready.


In [6]:
# CELL 6 - Cari filtered PCAP harian yang sudah valid
valid_day_parts = []
inspection_rows = []

for item in DAYS:
    day = item['day']
    attacker = item['attacker_ip']
    victim_private = item['victim_private']
    print('\n=== Checking candidates for', day, '===')
    chosen = None
    chosen_info = None
    for cand in item['filtered_candidates']:
        cand = Path(cand)
        info = pcap_contains_expected(cand, attacker, victim_private)
        inspection_rows.append({'day': day, 'candidate': str(cand), **info})
        print(cand)
        print('  exists:', info['exists'], 'packets:', info['packet_rows'], 'ips:', info['observed_ips'], 'valid_for_day:', info['valid_for_day'])
        if info['valid_for_day'] and chosen is None:
            chosen = cand
            chosen_info = info
    if chosen is not None:
        out_part = OUT_DIR / f'NOERROR_part_{day}_attacker_victim.pcap'
        shutil.copy2(chosen, out_part)
        valid_day_parts.append({'day': day, 'part_pcap': out_part, 'source': chosen, 'built_from_zip': False, 'inspect': chosen_info})
        print('Chosen existing valid part:', chosen)
    else:
        print('No valid existing filtered part for', day, '- will try build from ZIP in next cell.')

inspection_df = pd.DataFrame(inspection_rows)
display(inspection_df[['day','candidate','exists','packet_rows','observed_ips','valid_for_day','error']])
inspection_df.to_csv(OUT_DIR / 'NOERROR_candidate_filtered_pcap_inspection.csv', index=False)
print('Valid existing parts:', len(valid_day_parts))


=== Checking candidates for Wednesday-28-02-2018 ===
/content/drive/MyDrive/CICIDS2018/official_zero_day_infiltration_pcap_claim_ready/outputs/official_infiltration_attacker_victim_filtered.pcap
  exists: True packets: 1668 ips: ['13.58.225.34', '172.31.69.24'] valid_for_day: True
/content/drive/MyDrive/CICIDS2018/official_aws_infiltration_raw_pcap/outputs/official_infiltration_attacker_victim_filtered.pcap
  exists: True packets: 1668 ips: ['13.58.225.34', '172.31.69.24'] valid_for_day: True
/content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/filtered_Wednesday-28-02-2018_official_infiltration_attacker_victim_filtered.pcap
  exists: False packets: 0 ips: [] valid_for_day: False
Chosen existing valid part: /content/drive/MyDrive/CICIDS2018/official_zero_day_infiltration_pcap_claim_ready/outputs/official_infiltration_attacker_victim_filtered.pcap

=== Checking candidates for Thursday-01-03-2018 ===
/content/drive/MyDrive/CICIDS2018/official_in

,day,candidate,exists,packet_rows,observed_ips,valid_for_day,error
0,Wednesday-28-02-2018,/content/drive/MyDrive/CICIDS2018/official_zer...,True,1668,"[13.58.225.34, 172.31.69.24]",True,None
1,Wednesday-28-02-2018,/content/drive/MyDrive/CICIDS2018/official_aws...,True,1668,"[13.58.225.34, 172.31.69.24]",True,None
2,Wednesday-28-02-2018,/content/drive/MyDrive/CICIDS2018/official_inf...,False,0,[],False,file_missing_or_empty
3,Thursday-01-03-2018,/content/drive/MyDrive/CICIDS2018/official_inf...,True,1410,"[13.58.225.34, 172.31.69.13]",True,None
4,Thursday-01-03-2018,/content/drive/MyDrive/CICIDS2018/official_inf...,True,470,"[13.58.225.34, 172.31.69.13]",True,None


Valid existing parts: 2


In [7]:
# CELL 7 - Build missing day parts from ZIP jika perlu
built_rows = []
existing_days = {x['day'] for x in valid_day_parts}

for item in DAYS:
    day = item['day']
    if day in existing_days:
        print(day, 'already has valid part. Skip ZIP build.')
        continue

    zip_path = Path(item['zip_path'])
    attacker = item['attacker_ip']
    victim_private = item['victim_private']
    victims = item['victims']
    if not zip_path.exists() or zip_path.stat().st_size < EXPECTED_MIN_ZIP_BYTES:
        msg = f'ZIP tidak ditemukan atau terlihat partial: {zip_path}'
        status_line(False, msg)
        built_rows.append({'day': day, 'status': 'ZIP_MISSING', 'message': msg})
        continue

    print('\n=== Build from ZIP:', day, '===')
    try:
        with zipfile.ZipFile(zip_path) as zf:
            members = zf.infolist()
            victim_members = [m for m in members if victim_private in m.filename]
            if not victim_members:
                msg = f'Tidak ada member mengandung victim private {victim_private}'
                status_line(False, msg)
                built_rows.append({'day': day, 'status': 'MEMBER_NOT_FOUND', 'message': msg})
                continue
            print('Victim members:', [m.filename for m in victim_members])
            filtered_parts = []
            display_filter = f"(ip.addr == {attacker}) && (" + ' || '.join([f'ip.addr == {v}' for v in victims]) + ')'
            for m in victim_members:
                local_member = MEMBER_DIR / f'{day}__{Path(m.filename).name}'
                if not local_member.exists() or local_member.stat().st_size != m.file_size:
                    print('Extract member:', m.filename)
                    with zf.open(m) as src, open(local_member, 'wb') as dst:
                        shutil.copyfileobj(src, dst, length=1024*1024*16)
                else:
                    print('Member already extracted:', local_member)
                filtered_part = OUT_DIR / f'NOERROR_filtered_{day}_{local_member.stem}.pcap'
                if filtered_part.exists():
                    filtered_part.unlink()
                if not which('tshark'):
                    msg = 'tshark tidak tersedia untuk filtering'
                    status_line(False, msg)
                    built_rows.append({'day': day, 'status': 'TSHARK_MISSING', 'message': msg})
                    continue
                res = run_cmd(f'tshark -r "{local_member}" -Y "{display_filter}" -w "{filtered_part}"', check=False, capture=True)
                if res.returncode == 0 and filtered_part.exists() and filtered_part.stat().st_size > 0:
                    info = pcap_contains_expected(filtered_part, attacker, victim_private)
                    print('Filtered part info:', info)
                    if info['valid_for_day']:
                        filtered_parts.append(filtered_part)
                else:
                    print('Filter produced empty/failed:', filtered_part)

            if not filtered_parts:
                msg = 'Tidak ada filtered part valid dari ZIP'
                status_line(False, msg)
                built_rows.append({'day': day, 'status': 'FILTER_EMPTY', 'message': msg})
                continue

            day_part = OUT_DIR / f'NOERROR_part_{day}_attacker_victim.pcap'
            if day_part.exists():
                day_part.unlink()
            if len(filtered_parts) == 1:
                shutil.copy2(filtered_parts[0], day_part)
            else:
                input_args = ' '.join([f'"{p}"' for p in filtered_parts])
                res = run_cmd(f'mergecap -w "{day_part}" {input_args}', check=False, capture=True)
                if res.returncode != 0:
                    shutil.copy2(filtered_parts[0], day_part)
                    print('mergecap gagal; fallback copy filtered part pertama:', filtered_parts[0])
            info = pcap_contains_expected(day_part, attacker, victim_private)
            if info['valid_for_day']:
                valid_day_parts.append({'day': day, 'part_pcap': day_part, 'source': zip_path, 'built_from_zip': True, 'inspect': info})
                built_rows.append({'day': day, 'status': 'OK', 'part_pcap': str(day_part), 'packets': info['packet_rows'], 'observed_ips': info['observed_ips']})
            else:
                built_rows.append({'day': day, 'status': 'BUILT_BUT_INVALID', 'part_pcap': str(day_part), 'inspect': info})
    except Exception as e:
        msg = repr(e)
        status_line(False, f'Build failed for {day}: {msg}')
        built_rows.append({'day': day, 'status': 'EXCEPTION', 'message': msg})

built_df = pd.DataFrame(built_rows)
display(built_df)
built_df.to_csv(OUT_DIR / 'NOERROR_build_missing_day_parts_status.csv', index=False)
print('Total valid_day_parts:', len(valid_day_parts))
for p in valid_day_parts:
    print(p['day'], p['part_pcap'], p['inspect'].get('packet_rows'), p['inspect'].get('observed_ips'))

Wednesday-28-02-2018 already has valid part. Skip ZIP build.
Thursday-01-03-2018 already has valid part. Skip ZIP build.


""


Total valid_day_parts: 2
Wednesday-28-02-2018 /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/NOERROR_part_Wednesday-28-02-2018_attacker_victim.pcap 1668 ['13.58.225.34', '172.31.69.24']
Thursday-01-03-2018 /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/NOERROR_part_Thursday-01-03-2018_attacker_victim.pcap 1410 ['13.58.225.34', '172.31.69.13']


In [8]:
# CELL 8 - Merge final dari semua valid day parts
parts_by_day = {x['day']: x for x in valid_day_parts}
expected_days = [x['day'] for x in DAYS]
missing_days = [d for d in expected_days if d not in parts_by_day]

print('Expected days:', expected_days)
print('Available valid days:', sorted(parts_by_day.keys()))
print('Missing days:', missing_days)

final_status = {
    'ready': False,
    'missing_days': missing_days,
    'combined_final_pcap': str(COMBINED_FINAL_PCAP),
    'message': '',
}

if missing_days:
    final_status['message'] = 'NOT_READY: masih ada hari yang belum punya filtered PCAP valid.'
    print(final_status['message'])
else:
    pcaps_to_merge = [parts_by_day[d]['part_pcap'] for d in expected_days]
    if COMBINED_FINAL_PCAP.exists():
        COMBINED_FINAL_PCAP.unlink()
    if len(pcaps_to_merge) == 1:
        shutil.copy2(pcaps_to_merge[0], COMBINED_FINAL_PCAP)
    else:
        input_args = ' '.join([f'"{p}"' for p in pcaps_to_merge])
        res = run_cmd(f'mergecap -w "{COMBINED_FINAL_PCAP}" {input_args}', check=False, capture=True)
        if res.returncode != 0:
            final_status['message'] = 'NOT_READY: mergecap gagal.'
            print(final_status['message'])
        else:
            # copy juga ke nama kompatibel lama hanya jika final valid nanti
            print('Merge done:', COMBINED_FINAL_PCAP)

    if COMBINED_FINAL_PCAP.exists():
        final_info = inspect_pcap_ips(COMBINED_FINAL_PCAP)
        expected_private_victims = [x['victim_private'] for x in DAYS]
        observed = set(final_info.get('observed_ips', []))
        missing_victims = [ip for ip in expected_private_victims if ip not in observed]
        final_status.update({
            'packet_rows': final_info.get('packet_rows'),
            'observed_ips': final_info.get('observed_ips'),
            'missing_victims': missing_victims,
            'file_info': file_info(COMBINED_FINAL_PCAP, compute_sha256=True),
        })
        if not missing_victims and final_info.get('packet_rows', 0) > 0:
            final_status['ready'] = True
            final_status['message'] = 'READY: final PCAP memuat semua victim private IP resmi.'
            shutil.copy2(COMBINED_FINAL_PCAP, COMBINED_COMPAT_PCAP)
        else:
            final_status['message'] = f'NOT_READY: victim hilang pada PCAP final: {missing_victims}'

print(json.dumps(final_status, indent=2, default=str))
if STRICT_MODE and not final_status['ready']:
    raise RuntimeError(final_status['message'])

Expected days: ['Wednesday-28-02-2018', 'Thursday-01-03-2018']
Available valid days: ['Thursday-01-03-2018', 'Wednesday-28-02-2018']
Missing days: []

CMD: mergecap -w "/content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/official_infiltration_alldays_attacker_victim_filtered_FINAL.pcap" "/content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/NOERROR_part_Wednesday-28-02-2018_attacker_victim.pcap" "/content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/NOERROR_part_Thursday-01-03-2018_attacker_victim.pcap"
Merge done: /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/official_infiltration_alldays_attacker_victim_filtered_FINAL.pcap
{
  "ready": true,
  "missing_days": [],
  "combined_final_pcap": "/content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/official_infiltration_alldays_a

In [9]:
# CELL 9 - Ringkasan conversation dan 5-tuple jika final ready
flow_summary_df = pd.DataFrame()
if final_status.get('ready') and which('tshark'):
    print('Conversation IP:')
    run_cmd(f'tshark -r "{COMBINED_FINAL_PCAP}" -q -z conv,ip', check=False, capture=True)
    cmd = (
        f'tshark -r "{COMBINED_FINAL_PCAP}" -T fields '
        '-e frame.time_epoch -e frame.number -e ip.src -e ip.dst -e ip.proto '
        '-e tcp.srcport -e tcp.dstport -e udp.srcport -e udp.dstport '
        '-E header=y -E separator=, -E quote=d'
    )
    res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    from io import StringIO
    pkt_df = pd.read_csv(StringIO(res.stdout))
    def to_int_safe(v, default=-1):
        try:
            if pd.isna(v): return default
            return int(float(v))
        except Exception:
            return default
    def make_flow_key(row):
        proto_i = to_int_safe(row.get('ip.proto'))
        if proto_i == 6:
            sp, dp = to_int_safe(row.get('tcp.srcport')), to_int_safe(row.get('tcp.dstport'))
        elif proto_i == 17:
            sp, dp = to_int_safe(row.get('udp.srcport')), to_int_safe(row.get('udp.dstport'))
        else:
            sp, dp = -1, -1
        a = (str(row['ip.src']), sp)
        b = (str(row['ip.dst']), dp)
        return f'{a[0]}:{a[1]} <-> {b[0]}:{b[1]} proto={proto_i}' if a <= b else f'{b[0]}:{b[1]} <-> {a[0]}:{a[1]} proto={proto_i}'
    pkt_df['flow_key'] = pkt_df.apply(make_flow_key, axis=1)
    flow_summary_df = pkt_df.groupby('flow_key').agg(
        packets=('flow_key','size'),
        first_epoch=('frame.time_epoch','min'),
        last_epoch=('frame.time_epoch','max'),
    ).reset_index()
    flow_summary_df['duration_s'] = flow_summary_df['last_epoch'] - flow_summary_df['first_epoch']
    display(flow_summary_df)
    flow_summary_df.to_csv(OUT_DIR / 'NOERROR_FINAL_5tuple_summary.csv', index=False)
    print('Saved:', OUT_DIR / 'NOERROR_FINAL_5tuple_summary.csv')
else:
    print('Final PCAP belum ready atau tshark tidak tersedia. Lewati ringkasan 5-tuple.')

Conversation IP:

CMD: tshark -r "/content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/official_infiltration_alldays_attacker_victim_filtered_FINAL.pcap" -q -z conv,ip
STDOUT preview:
IPv4 Conversations
Filter:<No Filter>
                                               |       <-      | |       ->      | |     Total     |    Relative    |   Duration   |
                                               | Frames  Bytes | | Frames  Bytes | | Frames  Bytes |      Start     |              |
172.31.69.24         <-> 13.58.225.34             839 45 kB         829 239 kB       1668 285 kB        0.000000000     14059.0271
172.31.69.13         <-> 13.58.225.34             927 1,296 kB      483 54 kB        1410 1,351 kB   83533.556471000        71.3669

STDERR preview:
 Running as user "root" and group "root". This could be dangerous.



,flow_key,packets,first_epoch,last_epoch,duration_s
0,13.58.225.34:31337 <-> 172.31.69.13:50887 proto=6,1410,1.519913e+09,1.519913e+09,71.366900
1,13.58.225.34:31337 <-> 172.31.69.24:51603 proto=6,934,1.519829e+09,1.519834e+09,4995.038941
2,13.58.225.34:31337 <-> 172.31.69.24:54751 proto=6,734,1.519840e+09,1.519843e+09,3360.466711


Saved: /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/NOERROR_FINAL_5tuple_summary.csv


In [10]:
# CELL 10 - Simpan manifest dan command uji sistem
manifest = {
    'purpose': 'No-error final preparation of official CSE-CIC-IDS2018 all-days Infiltration PCAP for NIPS system test',
    'strict_mode': STRICT_MODE,
    'days': [
        {
            'day': x['day'],
            'attacker_ip': x['attacker_ip'],
            'victims': x['victims'],
            'victim_private': x['victim_private'],
            'zip_path': str(x['zip_path']),
            'official_windows_text': x['official_windows_text'],
        } for x in DAYS
    ],
    'valid_day_parts': [
        {
            'day': x['day'],
            'part_pcap': str(x['part_pcap']),
            'source': str(x['source']),
            'built_from_zip': x['built_from_zip'],
            'inspect': x['inspect'],
        } for x in valid_day_parts
    ],
    'final_status': final_status,
    'outputs': {
        'combined_final_pcap': str(COMBINED_FINAL_PCAP),
        'combined_compat_pcap': str(COMBINED_COMPAT_PCAP),
        'candidate_inspection_csv': str(OUT_DIR / 'NOERROR_candidate_filtered_pcap_inspection.csv'),
        'build_status_csv': str(OUT_DIR / 'NOERROR_build_missing_day_parts_status.csv'),
        'flow_summary_csv': str(OUT_DIR / 'NOERROR_FINAL_5tuple_summary.csv'),
    },
    'claim_boundary': (
        'Use this final PCAP for NIPS system test only if final_status.ready is true. '
        'It covers official attacker-victim traffic for both Infiltration days. '
        'For statistical all-flow class evaluation, report official TrafficForML CSV separately.'
    ),
    'created_at': datetime.datetime.now().isoformat(),
}
with open(MANIFEST_JSON, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)
print('Manifest saved:', MANIFEST_JSON)
print(json.dumps(manifest, indent=2, ensure_ascii=False, default=str)[:10000])

print('\n=== COMMAND UJI SISTEM ===')
if final_status.get('ready'):
    print('PCAP final:')
    print(COMBINED_FINAL_PCAP)
    print('\nReplay ke sensor NIPS:')
    print(f'tcpreplay -i <interface_ke_sensor> "{COMBINED_FINAL_PCAP}"')
    print('\nSnort offline -r:')
    print(f'snort -c /path/to/snort.lua -r "{COMBINED_FINAL_PCAP}" -A alert_fast -l /path/to/logdir')
else:
    print('PCAP final belum ready. Cek final_status dan CSV status di OUT_DIR.')

Manifest saved: /content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/official_infiltration_alldays_pcap_system_test_NOERROR_FINAL_manifest.json
{
  "purpose": "No-error final preparation of official CSE-CIC-IDS2018 all-days Infiltration PCAP for NIPS system test",
  "strict_mode": false,
  "days": [
    {
      "day": "Wednesday-28-02-2018",
      "attacker_ip": "13.58.225.34",
      "victims": [
        "18.221.148.137",
        "172.31.69.24"
      ],
      "victim_private": "172.31.69.24",
      "zip_path": "/content/drive/MyDrive/CICIDS2018/official_zero_day_infiltration_pcap_claim_ready/zips/Wednesday-28-02-2018_pcap.zip",
      "official_windows_text": [
        "10:50-12:05",
        "13:42-14:40"
      ]
    },
    {
      "day": "Thursday-01-03-2018",
      "attacker_ip": "13.58.225.34",
      "victims": [
        "18.216.254.154",
        "172.31.69.13"
      ],
      "victim_private": "172.31.69.13",
      "zip_path": "/content/drive

```text
READY: final PCAP memuat semua victim private IP resmi.
```

PCAP final yang dipakai untuk uji sistem:

```text
/content/drive/MyDrive/CICIDS2018/official_infiltration_alldays_pcap_system_test_ready/outputs/official_infiltration_alldays_attacker_victim_filtered_FINAL.pcap
```


Status `READY`:

> Sistem diuji menggunakan raw PCAP resmi CSE-CIC-IDS2018 yang mencakup traffic attacker-victim pada dua hari Infiltration resmi. Karena kelas Infiltration tidak digunakan pada pelatihan, validasi, maupun threshold tuning, pengujian ini digunakan sebagai skenario unseen attack berbasis raw traffic.